In [11]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import gradio as gr

In [12]:
df = pd.read_csv("content_based_recommendation_dataset.csv")

# Create user-friendly product column
price_col = None

for col in df.columns:
    if "price" in col.lower():
        price_col = col
        break

if price_col:
    df['Product'] = (
        df['Brand of the product'] +
        " | ₹" + df[price_col].astype(str) +
        " | Rating: " + df['Rating of the product'].astype(str)
    )
else:
    df['Product'] = (
        df['Brand of the product'] +
        " | Rating: " + df['Rating of the product'].astype(str)
    )

df.head()

,Number of clicks on similar products,Number of similar products purchased so far,Average rating given to similar products,Gender,Median purchasing price (in rupees),Rating of the product,Brand of the product,Customer review sentiment score (overall),Price of the product,Holiday,Season,Geographical locations,Probability for the product to be recommended to the person,Product
0,12,4,4.2,male,500,4.5,PUMA,0.8,200,No,winter,plains,0.9,PUMA | ₹500 | Rating: 4.5
1,8,2,3.8,female,3000,3.2,Lee,-0.4,300,Yes,monsoon,mountains,0.2,Lee | ₹3000 | Rating: 3.2
2,25,10,4.5,female,600,4.8,Head Hunters,0.6,1000,No,spring,plains,0.7,Head Hunters | ₹600 | Rating: 4.8
3,6,1,2.1,female,100,2.5,Johnson & Johnson,0.1,300,No,summer,coastal,0.1,Johnson & Johnson | ₹100 | Rating: 2.5
4,18,5,4.7,male,2000,4.3,Wakefit,0.9,700,Yes,winter,plains,0.8,Wakefit | ₹2000 | Rating: 4.3


In [13]:
def preprocess_data(df):
    df = df.copy()

    # One-hot encoding (IMPORTANT FIX)
    categorical_cols = [
        'Gender',
        'Brand of the product',
        'Season',
        'Geographical locations',
        'Holiday'
    ]

    df = pd.get_dummies(df, columns=[col for col in categorical_cols if col in df.columns])

    # Fill missing values
    df = df.fillna(0)

    return df

df_processed = preprocess_data(df)

In [14]:
def get_feature_matrix(df):
    features = [
        'clicks on similar products',
        'products purchased so far',
        'rating given to similar products',
        'purchasing price (in rupees)',
        'Rating of the product',
        'sentiment score (overall)'
    ]

    # Keep only available columns
    features = [col for col in features if col in df.columns]

    # Add encoded categorical columns
    encoded_cols = [col for col in df.columns if any(x in col for x in [
        'Brand of the product',
        'Gender',
        'Season',
        'Geographical locations',
        'Holiday'
    ])]

    features.extend(encoded_cols)

    return df[features]

feature_matrix = get_feature_matrix(df_processed)

In [15]:
scaler = MinMaxScaler()
feature_matrix_scaled = scaler.fit_transform(feature_matrix)

In [16]:
similarity_matrix = cosine_similarity(feature_matrix_scaled)

In [17]:
def recommend_by_product(product_name, top_n=5):
    if product_name not in df['Product'].values:
        return ["Product not found"]

    idx = df[df['Product'] == product_name].index[0]
    selected_brand = df.iloc[idx]['Brand of the product']

    scores = list(enumerate(similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    seen = set()
    recommendations = []

    for i, score in scores:
        if i == idx:
            continue

        product = df.iloc[i]['Product']
        brand = df.iloc[i]['Brand of the product']

        # 🚫 Skip SAME BRAND
        if brand == selected_brand:
            continue

        # 🚫 Skip duplicates
        if product in seen:
            continue

        seen.add(product)
        recommendations.append(product)

        if len(recommendations) == top_n:
            break

    return recommendations

In [18]:
def get_recommendations(product_name, top_n):
    try:
        recs = recommend_by_product(product_name, int(top_n))
        return ", ".join(recs)
    except:
        return "Invalid Input"

interface = gr.Interface(
    fn=get_recommendations,
    inputs=[
        gr.Dropdown(
            choices=df['Product'].unique().tolist(),
            label="Select Product"
        ),
        gr.Slider(1, 10, step=1, label="Top N Recommendations")
    ],
    outputs=gr.Textbox(label="Recommended Products"),
    title="E-commerce Recommendation System",
    description="Select a product to get similar product recommendations"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
